In [ ]:
# Evaluate on the 3.20 test set with bootstrap sampling.
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, precision_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import random
from Bio import SeqIO

In [ ]:
import argparse
import pandas as pd
import lightgbm as lgb
import joblib
from sklearn.preprocessing import StandardScaler
import copy
# import random
import itertools
from sklearn.impute import SimpleImputer
import numpy as np

In [ ]:
def preprocess_data(df):
    # scaler = StandardScaler()
    # data_scaled = scaler.fit_transform(data)
    # Count the frequency of k-mer in each RNA sequence
        # k-mer was normalized by total k-mer count of each RNA sequence
    def _count_kmer(Dataset, k):  # k = 3, 4, 5
        
        # copy dataset
        dataset = copy.deepcopy(Dataset)
        # alphabet of nucleotide
        nucleotide = ['A', 'C', 'G', 'T']
        
        # generate k-mers
        #  k == 5:
        five = list(itertools.product(nucleotide, repeat=5))
        pentamer = [''.join(n) for n in five]
        
        #  k == 4:
        four = list(itertools.product(nucleotide, repeat=4))
        tetramer = [''.join(n) for n in four]

        # k == 3:
        three = list(itertools.product(nucleotide, repeat=3))
        threemer = [''.join(n) for n in three]
        
        # input features can be combinations of different k values
        if k == 34:
            table_kmer = dict.fromkeys(threemer, 0)
            table_kmer.update(dict.fromkeys(tetramer, 0))
        elif k == 45:
            table_kmer = dict.fromkeys(tetramer, 0)
            table_kmer.update(dict.fromkeys(pentamer, 0))
        elif k == 345:
            table_kmer = dict.fromkeys(threemer, 0)
            table_kmer.update(dict.fromkeys(tetramer, 0))
            table_kmer.update(dict.fromkeys(pentamer, 0))

        # count k-mer for each sequence
        for mer in table_kmer.keys():
            table_kmer[mer] = dataset["sequence"].apply(lambda x: x.count(mer))
        
        # for k-mer raw count without normalization, index: nuc:1 or cyto:0
        rawcount_kmer_df = pd.DataFrame(table_kmer)
        df1_rawcount = pd.concat([rawcount_kmer_df, dataset["name"]], axis=1)
        df1_rawcount.index = dataset["tag"]

        # for k-mer frequency with normalization, 
        freq_kmer_df = rawcount_kmer_df.apply(lambda x: x / x.sum(), axis=1)
        df1 = pd.concat([freq_kmer_df, dataset["name"]], axis=1)
        df1.index = dataset["tag"]

        return df1, df1_rawcount

    df_kmer_test, df_kmer_test_raw = _count_kmer(df, 345)
    del df_kmer_test['name']
    x_kmer = df_kmer_test.values
    imputer = SimpleImputer(strategy='mean')
    x_test = imputer.fit_transform(x_kmer)
    # y_test = np.array(df_kmer_test.index)

    return x_test

In [ ]:
def load_model(model_path):
    model = joblib.load(model_path)
    return model

In [ ]:
def predict(model, x_test, df):
    y_pred = model.predict(x_test)
    y_prob = model.predict_proba(x_test)[:,1]
    df['prediction'] = y_pred  # Add the predicted-label column.
    df['prob'] = y_prob  # Add the predicted-probability column.

    evaluate_df = df
    # Report observed and predicted tag counts.
    true_counts = evaluate_df['tag'].value_counts().to_dict()
    pred_counts = evaluate_df['prediction'].value_counts().to_dict()
    print(f"True tag counts: {true_counts}")
    print(f"Predicted tag counts: {pred_counts}")
    auroc = roc_auc_score(evaluate_df["tag"], evaluate_df["prob"])
    accuracy = accuracy_score(evaluate_df['tag'], evaluate_df['prediction'])
    sensitivity = recall_score(evaluate_df['tag'], evaluate_df['prediction'])  # Sensitivity = True Positive Rate
    specificity = recall_score(evaluate_df['tag'], evaluate_df['prediction'], pos_label=0)  # Specificity = True Negative Rate
    f1 = f1_score(evaluate_df['tag'], evaluate_df['prediction'])
    print(f"AUROC: {auroc:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    return evaluate_df

In [ ]:
def parse_fasta_to_dataframe(fasta_path):
    records = []
    with open(fasta_path, 'r') as file:
        name, tag, seq_lines = None, None, []
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                if name:
                    sequence = ''.join(seq_lines)
                    records.append({'name': name, 'tag': tag, 'sequence': sequence})
                
                header = line[1:]
                # Split once from the right by "_".
                parts = header.rsplit('_', 1)
                if len(parts) == 2:
                    name = parts[0]
                    try:
                        tag = int(parts[1])
                    except ValueError:
                        tag = parts[1]
                else:
                    name = header
                    tag = None
                seq_lines = []
            else:
                seq_lines.append(line)
                
        # Add the final record.
        if name:
            sequence = ''.join(seq_lines)
            records.append({'name': name, 'tag': tag, 'sequence': sequence})
    
    return pd.DataFrame(records)

In [ ]:
model = load_model('../../models/ML_models/circRNA_ML_Model_tridivided_intra5fold_Output/RandomForest/best_RandomForest_model.pkl')
df = parse_fasta_to_dataframe('../../sample_preprocessing/circRNA_external_testset/merged_dataset.fasta')

In [ ]:
x_test = preprocess_data(df)
evaluate_df = predict(model, x_test, df)

In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

# Draw the confusion matrix with seaborn.
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Cellular', 'EV'], yticklabels=['Cellular', 'EV'])
plt.title('circExor')
plt.ylabel('Acurate label')
plt.xlabel('Predicted label')
plt.show()

In [ ]:
import os
import random
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, matthews_corrcoef

random.seed(100)

model = load_model('../../models/ML_models/circRNA_ML_Model_tridivided_intra5fold_Output/RandomForest/best_RandomForest_model.pkl')
df = parse_fasta_to_dataframe('../../sample_preprocessing/circRNA_external_testset/merged_dataset.fasta')

# Define bootstrap parameters.
n_bootstraps = 5  # Number of bootstrap subsets.
sample_size = 400  # Sample size per bootstrap subset.
output_dir = './bootstrap_results'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

all_evaluate_dfs = []  # Collect predictions for final merging.

for i in range(n_bootstraps):
    # Generate a bootstrap subset with replacement.
    bootstrap_df = df.sample(n=sample_size, replace=True, random_state=100 + i).copy()
    
    # Preprocess and extract features.
    x_test = preprocess_data(bootstrap_df)
    
    # Run prediction and merge results.
    current_evaluate_df = predict(model, x_test, bootstrap_df)
    
    # Ensure the prediction column exists.
    if 'prediction' not in current_evaluate_df.columns:
        current_evaluate_df['prediction'] = (current_evaluate_df['prob'] >= 0.5).astype(int)
    
    all_evaluate_dfs.append(current_evaluate_df)  # Append to the result list.
    
    # Store each DataFrame under a dynamic variable name.
    globals()[f'evaluate_df_{i+1}'] = current_evaluate_df
    
    # Save predictions for the current subset.
    save_path = os.path.join(output_dir, f'bootstrap_set_{i+1}_predictions.csv')
    current_evaluate_df.to_csv(save_path, index=False)
    print(f"Bootstrap set {i+1} saved to {save_path} and loaded as variable 'evaluate_df_{i+1}'")
    
    y_true = current_evaluate_df['tag'].astype(int)
    y_prob = current_evaluate_df['prob']
    y_pred = current_evaluate_df['prediction'].astype(int)
    
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    print(f"--- Bootstrap {i+1} Metrics ---")
    print(f"AUROC:    {auroc:.4f}")
    print(f"AUPRC:    {auprc:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"MCC:      {mcc:.4f}\n")

In [ ]:
# Merge all predictions.
merged_evaluate_df = pd.concat(all_evaluate_dfs, ignore_index=True)

# Calculate the overall confusion matrix.
cm = confusion_matrix(merged_evaluate_df['tag'].astype(int), merged_evaluate_df['prediction'].astype(int))

# Draw the confusion matrix with seaborn.
plt.figure(figsize=(6, 5))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],
                 annot_kws={'size': 16, 'weight': 'bold'},  # Adjust annotation font size and weight.
                 cbar_kws={'shrink': 0.8})  # Adjust the colorbar size.

plt.title('circExor', fontsize=18, fontweight='bold')
plt.ylabel('Acurate label', fontsize=16)
plt.xlabel('Predicted label', fontsize=16)
plt.xticks(fontsize=15, fontweight='bold')
plt.yticks(fontsize=15, fontweight='bold')

# Adjust colorbar tick label size.
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=14)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as st

# Metric names and model list.
metrics = ['AUROC', 'AUPRC', 'Accuracy', 'F1 Score', 'MCC']
models_names = ['circExor']  # Reserved for future grouped bar plots.

# Metric matrix by model.
model_data_dict = {
    'circExor': np.array([
        [0.7050, 0.6650, 0.6550, 0.6683, 0.3094],
        [0.6993, 0.6513, 0.6400, 0.6382, 0.2805],
        [0.6446, 0.5909, 0.6000, 0.5939, 0.2001],
        [0.6786, 0.6542, 0.6375, 0.6523, 0.2750],
        [0.6748, 0.5979, 0.6100, 0.5916, 0.2185]
    ])
    # Add new models here if needed.
    # 'exoGRU': np.array([...]),
}

# Warm color palette from light to dark.
warm_colors = ['#F8D7B4', '#F0C9A8', '#E8BB99', '#D89F7B', '#C8835D', '#B05930']

# Assign colors to models.
# Use additional colors when more models are added.
model_colors = [warm_colors[3]] 

# Set plot parameters.
fig, ax = plt.subplots(figsize=(10, 6))
n_metrics = len(metrics)
n_models = len(models_names)

# Calculate bar width and offsets from the number of models.
bar_width = 0.8 / (n_models + 0.5) if n_models > 1 else 0.4
x = np.arange(n_metrics)

max_y = 0  # Track the required y-axis maximum.

# Draw grouped bars for each model.
for i, model_name in enumerate(models_names):
    data = model_data_dict[model_name]
    
    # Calculate means and 95% confidence intervals.
    means = np.mean(data, axis=0)
    stds = np.std(data, axis=0, ddof=1)
    n = data.shape[0]
    cis = st.t.ppf(0.975, n-1) * (stds / np.sqrt(n))
    
    # Calculate grouped offsets.
    offset = (i - n_models/2 + 0.5) * bar_width
    
    # Draw bars for this model across metrics.
    bars = ax.bar(x + offset, means, bar_width, 
                  label=model_name, color=model_colors[i], 
                  edgecolor='black', linewidth=1.2, alpha=0.85,
                  yerr=cis, capsize=8, error_kw={'linewidth': 1.5, 'capthick': 1.5})
    
    # Add value labels above bars.
    for j, bar in enumerate(bars):
        height = bar.get_height()
        current_top = height + cis[j]
        if current_top > max_y:
            max_y = current_top
            
        y_pos = current_top + 0.015
        ax.text(bar.get_x() + bar.get_width()/2., y_pos,
                f'{means[j]:.3f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Set plot attributes.
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=16, fontweight='bold')
ax.tick_params(axis='y', labelsize=14)

# Leave space for top labels.
ax.set_ylim(0, max_y * 1.15) 

ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylabel('Score', fontsize=16, fontweight='bold')
ax.set_title('circExor Performance (5 Bootstraps with 95% CI)', fontsize=18, fontweight='bold')

# Show the legend when labels are available.
ax.legend(fontsize=14, loc='upper right')

# Adjust layout.
plt.tight_layout()
plt.show()